# Практика 1. Графовые данные и baseline

Сегодня мы:
1. научимся загружать графы и смотреть на них — `networkx` и `PyTorch Geometric`;
2. посчитаем классические признаки вершин и обучим на них обычный классификатор;
3. реализуем Weisfeiler–Lehman kernel и классифицируем молекулы.

Всё, что мы получим сегодня, — это **baseline**. В следующих практиках мы будем
пытаться его побить нейросетями, и получится это не всегда.

Ячейки помечены `# TODO` — их нужно дописать. Разбор — на занятии.

In [ ]:
# В Colab первая ячейка ставит зависимости (2-3 минуты)
try:
    import torch_geometric  # noqa: F401
except ImportError:
    !pip install -q torch_geometric
!pip install -q networkx scikit-learn matplotlib

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.svm import SVC

np.random.seed(0)
torch.manual_seed(0)
print("готово")

## 1. Загружаем граф

`Cora` — граф цитирования научных статей: вершина это статья, ребро — ссылка одной
статьи на другую. У каждой статьи есть признаки (какие слова в ней встречаются) и класс (тематика).

In [ ]:
from torch_geometric.datasets import Planetoid

dataset = Planetoid(root="/tmp/Cora", name="Cora")
data = dataset[0]
print(data)

In [ ]:
# TODO: выведите основные характеристики графа
# число вершин, число рёбер (учтите, что edge_index хранит каждое ребро дважды),
# размерность признаков, число классов, сколько вершин размечено для обучения
n_nodes = ...
n_edges = ...
n_features = ...
n_classes = ...
n_train = ...

print(f"вершин: {n_nodes}, рёбер: {n_edges}, признаков: {n_features}")
print(f"классов: {n_classes}, размечено для обучения: {n_train}")

Обратите внимание: размечено всего 140 вершин из 2708 — это 5% графа.
Именно поэтому структура связей так важна: меток мало, а связей много.

## 2. Смотрим на граф глазами

`PyG` хранит граф как `edge_index`. Для алгоритмов и визуализации удобнее `networkx`.

In [ ]:
from torch_geometric.utils import to_networkx

G = to_networkx(data, to_undirected=True)
print(G)
print("связных компонент:", nx.number_connected_components(G))
print("размер наибольшей компоненты:", len(max(nx.connected_components(G), key=len)))

In [ ]:
# посмотрим на кусочек графа: возьмём одну вершину и всех её соседей в двух шагах
center = int(np.argmax([G.degree(v) for v in G.nodes()]))
around = nx.ego_graph(G, center, radius=2)
colors = ["#E07A3F" if v == center else "#1F3A5F" for v in around.nodes()]

plt.figure(figsize=(7, 4.5))
pos = nx.spring_layout(around, seed=3)
nx.draw_networkx_edges(around, pos, alpha=0.3, edge_color="#8B97A8")
nx.draw_networkx_nodes(around, pos, node_color=colors, node_size=60, linewidths=0)
plt.title(f"Окрестность самой связной статьи: {around.number_of_nodes()} вершин")
plt.axis("off"); plt.show()

In [ ]:
# G.degree() возвращает пары (вершина, степень)
# TODO: соберите массив степеней всех вершин
degrees = np.array([...])

print("средняя степень:", degrees.mean().round(2))
print("максимальная степень:", degrees.max())

In [ ]:
# график уже написан — просто посмотрите на форму распределения
plt.figure(figsize=(7, 3))
plt.hist(degrees, bins=range(1, degrees.max() + 2), color="#1F3A5F")
plt.yscale("log")
plt.axvline(degrees.mean(), color="#E07A3F", linestyle="--", linewidth=2)
plt.xlabel("степень вершины"); plt.ylabel("число вершин (log)")
plt.title("Распределение степеней в Cora"); plt.show()

Распределение степеней тяжёлохвостое: большинство статей цитируют единицы, а есть
обзоры с сотнями ссылок. Это типично для реальных графов и объясняет, почему в GCN
нормируют вклад соседа на его степень.

## 3. Гомофилия

Гомофилия — доля рёбер, соединяющих вершины одного класса. От неё зависит,
будет ли вообще работать графовая модель.

In [ ]:
# TODO: посчитайте гомофилию графа Cora
# доля рёбер, у которых data.y[u] == data.y[v]
y = data.y.numpy()
edge_index = data.edge_index.numpy()

homophily = ...
print("гомофилия Cora:", round(homophily, 3))

In [ ]:
# нарисуем, как гомофилия выглядит по классам
classes = sorted(set(y))
inside = []
for c in classes:
    mask = y[edge_index[0]] == c
    inside.append((y[edge_index[1]][mask] == c).mean())

plt.figure(figsize=(7, 3))
plt.bar([str(c) for c in classes], inside, color="#1F3A5F")
plt.axhline(1 / len(classes), color="#E07A3F", linestyle="--",
            label="уровень случайного графа")
plt.xlabel("класс статьи"); plt.ylabel("доля соседей того же класса")
plt.legend(frameon=False); plt.show()

Около 0.81 — очень высокая гомофилия. Статьи цитируют статьи своей же тематики.
В лекции 4 мы увидим, что бывает, когда это не так.

## 4. Классические признаки вершин

Соберём признаки, не заглядывая в `data.x`: только структура графа.

Три признака посчитает `networkx` — это готовые функции. Четвёртый, степень, посчитаем сами,
чтобы было видно, что за этим ничего сложного не стоит.

In [ ]:
# TODO: посчитайте степень каждой вершины, пройдя по списку рёбер
# edge_index[0] — откуда идёт ребро, edge_index[1] — куда
# каждое ребро в Cora записано дважды, поэтому считаем только по первой строке
deg = np.zeros(n_nodes)
for u in data.edge_index[0].numpy():
    ...

# проверим себя: должно совпасть с networkx
assert (deg == np.array([G.degree(v) for v in range(n_nodes)])).all()
print("степени посчитаны верно")

In [ ]:
# остальные признаки берём готовыми из networkx
clust = np.array(list(nx.clustering(G).values()))       # доля связей между соседями
pr = np.array(list(nx.pagerank(G).values()))            # важность вершины
core = np.array(list(nx.core_number(G).values()))       # глубина вложенности в плотное ядро

X_struct = np.column_stack([deg, clust, pr, core])
print("матрица признаков:", X_struct.shape)

In [ ]:
def evaluate(X, y, name):
    """Честный замер: 5 фолдов, стратификация, среднее и разброс."""
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
    scores = cross_val_score(LogisticRegression(max_iter=2000), X, y, cv=cv)
    print(f"{name}: accuracy {scores.mean():.3f} ± {scores.std():.3f}")
    return scores.mean()

X_norm = (X_struct - X_struct.mean(0)) / (X_struct.std(0) + 1e-9)
acc_struct = evaluate(X_norm, y, "структурные признаки")

In [ ]:
# TODO: обучите тот же классификатор на признаках самих статей (data.x)
# и сравните с результатом на структурных признаках
acc_content = ...

# TODO: а теперь объедините оба набора признаков
acc_both = ...

**Вопрос для обсуждения.** Почему структурные признаки дают так мало?
Вспомните клуб карате из лекции: центральности говорят, насколько вершина важна,
но ничего не говорят о том, к какому классу она принадлежит.

Заметьте также: здесь мы использовали 5-фолдовую кросс-валидацию по всем вершинам,
а стандартный протокол Cora — 140 размеченных вершин. Числа из статей с этими
сравнивать нельзя.

## 5. Weisfeiler–Lehman kernel

Теперь перейдём к задаче уровня графа: классификация молекул `NCI1` — активен ли препарат
против раковых клеток. Реализуем WL-kernel из лекции.

Датасет заметно больше `MUTAG` из лекции: 4110 молекул вместо 188. Считаться будет
около минуты.

In [ ]:
from torch_geometric.datasets import TUDataset

# NCI1: молекулы из скрининга противораковой активности
# вершина — атом, ребро — химическая связь, метка графа — активен ли препарат
nci = TUDataset(root="/tmp/NCI1", name="NCI1")
print(nci)
print("графов:", len(nci), "| классов:", nci.num_classes)
print("пример:", nci[0])

In [ ]:
# TODO: реализуйте одну итерацию WL
# на вход: граф networkx и текущие метки {вершина: метка}
# на выход: новые метки — хеш от своей метки и отсортированных меток соседей
def wl_iteration(graph, labels):
    new_labels = {}
    for v in graph.nodes():
        ...
    return new_labels

In [ ]:
# TODO: соберите вектор признаков графа — гистограмму WL-меток за k итераций
# Метки со всех итераций складываются в один словарь «метка -> сколько раз встретилась»
from collections import Counter

def wl_features(graph, initial_labels, iterations=3):
    counts = Counter()
    labels = dict(initial_labels)
    ...
    return counts

In [ ]:
def graph_to_nx(d):
    g = nx.Graph()
    g.add_nodes_from(range(d.num_nodes))
    g.add_edges_from(d.edge_index.t().tolist())
    # начальная метка вершины — тип атома (one-hot в d.x)
    labels = {v: str(int(d.x[v].argmax())) for v in range(d.num_nodes)}
    return g, labels

graphs = [graph_to_nx(d) for d in nci]
y_nci = np.array([int(d.y) for d in nci])
print("первый граф:", graphs[0][0])

In [ ]:
def counters_to_matrix(counters):
    """Превращает список словарей «метка -> сколько раз» в матрицу признаков.

    Техническая функция: собирает общий словарь меток и раскладывает по колонкам.
    """
    vocabulary = sorted({key for c in counters for key in c})
    index = {k: i for i, k in enumerate(vocabulary)}
    matrix = np.zeros((len(counters), len(vocabulary)))
    for i, c in enumerate(counters):
        for k, v in c.items():
            matrix[i, index[k]] = v
    return matrix

all_counts = [wl_features(g, lab, iterations=3) for g, lab in graphs]
X_wl = counters_to_matrix(all_counts)
print("матрица WL-признаков:", X_wl.shape)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
scores = cross_val_score(SVC(kernel="linear"), X_wl, y_nci, cv=cv)
print(f"WL-kernel + SVM на NCI1: {scores.mean():.3f} ± {scores.std():.3f}")

# для сравнения: классификация по простейшим признакам графа
X_simple = np.array([[g.number_of_nodes(), g.number_of_edges()] for g, _ in graphs])
scores_simple = cross_val_score(SVC(kernel="linear"), X_simple, y_nci, cv=cv)
print(f"только размер графа:      {scores_simple.mean():.3f} ± {scores_simple.std():.3f}")

### Что означают эти два числа

WL-kernel должен уверенно обойти признаки размера: примерно 0.82 против 0.63.

Тривиальный baseline здесь не проходит — и это как раз правильная ситуация: значит,
задача действительно про структуру молекулы, а не про её размер. Но посчитать его
всё равно было обязательно: пока вы не увидели 0.63, вы не знаете, сколько стоит ваш 0.82.

In [ ]:
# TODO: проверьте, добавляет ли размер графа что-то сверх WL
# обучите модель на объединённых признаках [X_wl, X_simple] и сравните с WL
X_both = ...
scores_both = ...
print(f"WL + размер: {scores_both.mean():.3f} ± {scores_both.std():.3f}")

In [ ]:
# соберём все сегодняшние замеры на одну картинку
labels = ["структура", "слова", "слова + структура", "WL (NCI1)", "размер (NCI1)"]
values = [acc_struct, acc_content, acc_both, scores.mean(), scores_simple.mean()]
colors = ["#6B7280", "#1F3A5F", "#1F3A5F", "#E07A3F", "#6B7280"]

plt.figure(figsize=(8, 3.4))
bars = plt.bar(labels, values, color=colors)
for rect, v in zip(bars, values):
    plt.text(rect.get_x() + rect.get_width() / 2, v + 0.01, f"{v:.3f}",
             ha="center", fontweight="bold")
plt.ylim(0, 1.0); plt.ylabel("accuracy")
plt.title("Все baseline сегодняшней практики")
plt.xticks(rotation=15, ha="right"); plt.tight_layout(); plt.show()

## 6. Выводы

- Загрузили граф, посмотрели на степени, компоненты и гомофилию.
- Структурные признаки вершины сами по себе слабы: они описывают роль, а не класс.
- WL-kernel на NCI1 уверенно обходит признаки размера: структура молекулы важнее её величины.

**Это наш baseline.** В следующих практиках мы будем учить представления вместо того,
чтобы придумывать признаки руками — и каждый раз возвращаться к этим числам, чтобы
проверить, есть ли выигрыш.

### Задание для самостоятельной работы
1. Посчитайте гомофилию для `CiteSeer` и `PubMed`. Где она выше?
2. Попробуйте WL с 1, 2, 5 итерациями — как меняется качество на MUTAG?
3. Сравните WL-kernel с `graphlet`-признаками: сколько в каждой молекуле треугольников
   и циклов длины 4 (`nx.cycle_basis`).
4. Повторите тот же эксперимент на `MUTAG` (188 молекул). Вас ждёт сюрприз: там счётчик
   вершин и рёбер обходит WL-kernel. Подумайте, что это говорит о датасете — и почему
   выводы, полученные на одном маленьком наборе, нельзя переносить на другие.